# 11. Устойчивость прогноза на новых соревнованиях

Повторяем разбиение по соревнованиям с разными seed. Все команды одного соревнования остаются только в train или только в test. Сравниваем правило по опыту, константу, логистическую регрессию и LightGBM.

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from src.build_dataset import get_dataset, FEATURES, TARGET
from src.modeling import baseline_models

df = get_dataset()
seeds = [17, 29, 42, 73, 101]
rows = []
for seed in seeds:
    split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(split.split(df, groups=df["CompetitionId"]))
    train, test = df.iloc[train_idx], df.iloc[test_idx]
    X_train, y_train = train[FEATURES], train[TARGET]
    X_test, y_test = test[FEATURES], test[TARGET]
    predictions = {"experience_rule": X_test["max_prev_comps"].fillna(0).to_numpy()}
    for name, model in baseline_models().items():
        model.fit(X_train, y_train)
        predictions[name] = model.predict_proba(X_test)[:, 1]
    model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                           is_unbalance=True, random_state=42, verbose=-1, n_jobs=4)
    model.fit(X_train, y_train)
    predictions["lightgbm"] = model.predict_proba(X_test)[:, 1]
    for name, score in predictions.items():
        rows.append({"seed": seed, "model": name,
                     "test_competitions": test["CompetitionId"].nunique(),
                     "train_rows": len(train), "test_rows": len(test),
                     "roc_auc": roc_auc_score(y_test, score),
                     "pr_auc": average_precision_score(y_test, score),
                     "test_top10_rate": y_test.mean()})

results = pd.DataFrame(rows)
results.to_csv("../reports/model_stability_by_split.csv", index=False, float_format="%.6f")
results.groupby("model")[["roc_auc", "pr_auc"]].agg(["mean", "std", "min", "max"]).round(4)

ROC-AUC and PR-AUC are reported for each held-out split. The output CSV preserves per-seed scores so the summary can be checked or plotted.

In [ ]:
summary = results.groupby("model")[["roc_auc", "pr_auc"]].agg(["mean", "std", "min", "max"])
ax = summary["roc_auc"][ ["mean", "std"] ].plot.bar(yerr="std", capsize=4, rot=20,
    title="ROC-AUC across held-out competition splits", ylabel="ROC-AUC", ylim=(0.45, 0.70))
ax.figure.tight_layout()
ax.figure.savefig("../reports/figures/11_model_stability.png", dpi=160)
